# SAM Image Encoder Profiler — SAM1 / SAM2 / SAM3

Profiles the **vision encoder** of each SAM generation side-by-side.

| Model | Backbone | Layers | Hidden | Patch | Tokens (default res) | Window |
|---|---|---|---|---|---|---|
| SAM1 (ViT-H) | Plain ViT | 32 | 1280 | 16 | 64×64 = 4096 | 14, global @ [7,15,23,31] |
| SAM2 (Hiera-L) | Hiera | 2+6+36+4 | 144→1152 | 4 (stride) | 256→16 multi-scale | per-stage |
| SAM3 | ViT + RoPE | 32 | 1024 | 14 | 72×72 = 5184 | 24, global @ [7,15,23,31] |

**What is measured per model:**
- Patch embedding
- Per-layer: `layer_norm1 | attention | layer_norm2 | mlp`
- Attention map shape (window vs global)
- Neck / FPN
- Window-vs-global attention cost ratio

**Requirements:**
- SAM1 / SAM2 weights are public — no token needed
- SAM3 is gated — pass your token in `HF_TOKEN` below, or set `SAM3_MOCK = True`

## 0. Configuration

In [1]:
import torch

DEVICE   = "cuda:0"
DTYPE    = torch.bfloat16   # bfloat16 | float16 | float32
REPEATS  = 10               # timing repeats per model

# ── SAM1 ────────────────────────────────────────────────────────────────────
SAM1_MODEL  = "facebook/sam-vit-large"   # sam-vit-huge | sam-vit-large | sam-vit-base
SAM1_IMSIZE = 1024                      # pixels

# ── SAM2 ────────────────────────────────────────────────────────────────────
SAM2_MODEL  = "facebook/sam2-hiera-large"  # sam2-hiera-tiny/small/base-plus/large
SAM2_IMSIZE = 1024

# ── SAM3 ────────────────────────────────────────────────────────────────────
SAM3_MODEL  = "facebook/sam3"   # facebook/sam3 | facebook/sam3.1
SAM3_IMSIZE = 1008              # model default: 1008 px
HF_TOKEN    = None              # set to "hf_..." if you have access
SAM3_MOCK   = True             # True = random weights, no token needed

print(f"Device: {DEVICE}  |  dtype: {DTYPE}")
print(f"CUDA: {torch.cuda.get_device_name(DEVICE)}")

Device: cuda:0  |  dtype: torch.bfloat16
CUDA: NVIDIA RTX A4000


## 1. Shared Utilities

In [2]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
import torch.nn as nn

# ── Stats accumulator ────────────────────────────────────────────────────────

@dataclass
class Stats:
    latencies_ms: List[float] = field(default_factory=list)
    peak_mem_mb: float = 0.0

    @property
    def mean(self): return sum(self.latencies_ms) / max(len(self.latencies_ms), 1)
    @property
    def mn(self):   return min(self.latencies_ms) if self.latencies_ms else 0.0
    @property
    def mx(self):   return max(self.latencies_ms) if self.latencies_ms else 0.0


# ── CUDA-event hook timer ────────────────────────────────────────────────────

class HookTimer:
    """Wraps a module with pre/post forward hooks to measure GPU time."""
    def __init__(self, module: nn.Module, stats: Stats, device: torch.device):
        self.stats  = stats
        self.device = device
        self._pre  = module.register_forward_pre_hook(self._pre_hook)
        self._post = module.register_forward_hook(self._post_hook)

    def _pre_hook(self, module, args):
        torch.cuda.synchronize(self.device)
        torch.cuda.reset_peak_memory_stats(self.device)
        self._mem0 = torch.cuda.memory_allocated(self.device)
        self._s    = torch.cuda.Event(enable_timing=True)
        self._e    = torch.cuda.Event(enable_timing=True)
        self._s.record()

    def _post_hook(self, module, args, output):
        self._e.record()
        torch.cuda.synchronize(self.device)
        self.stats.latencies_ms.append(self._s.elapsed_time(self._e))
        peak = torch.cuda.max_memory_allocated(self.device)
        self.stats.peak_mem_mb = max(self.stats.peak_mem_mb,
                                     max(0, peak - self._mem0) / 1024**2)

    def remove(self):
        self._pre.remove()
        self._post.remove()


def attach_timers(modules_dict: Dict[str, nn.Module],
                  device: torch.device) -> Dict[str, tuple]:
    """Attach HookTimers to a {name: module} dict. Returns {name: (stats, timer)}."""
    result = {}
    for name, mod in modules_dict.items():
        s = Stats()
        t = HookTimer(mod, s, device)
        result[name] = (s, t)
    return result


def detach_timers(timers_dict: Dict[str, tuple]):
    for _, (_, t) in timers_dict.items():
        t.remove()


# ── Pretty-print table ───────────────────────────────────────────────────────

def print_table(title: str, rows: List[tuple], total_ms: float, W: int = 82):
    """rows: list of (name, stats_or_ms, indent, extra_str)"""
    print(f"\n{'='*W}")
    print(f"  {title}")
    print(f"{'='*W}")
    print(f"  {'Component':<32} {'Mean ms':>9} {'Min ms':>9} {'Max ms':>9} {'Mem MB':>9} {'Share':>7}")
    print(f"  {'─'*(W-4)}")
    for (name, s, indent, extra) in rows:
        pad = "  " * indent
        if isinstance(s, Stats):
            mean, mn, mx, mem = s.mean, s.mn, s.mx, s.peak_mem_mb
        else:
            mean = mn = mx = s; mem = 0.0
        pct = 100 * mean / total_ms if total_ms else 0
        tag = f"  {extra}" if extra else ""
        print(f"  {pad}{name:<{32-2*indent}} {mean:>9.2f} {mn:>9.2f} {mx:>9.2f} "
              f"{mem:>9.1f} {pct:>6.1f}%{tag}")
    print(f"  {'─'*(W-4)}")
    print(f"  {'TOTAL':<32} {total_ms:>9.2f}")
    print(f"  Throughput: {1000/total_ms:.2f} fps")

print("Utilities ready.")

Utilities ready.


## 2. SAM1 — Plain ViT Image Encoder

Architecture: **32-layer ViT** with absolute position embeddings.  
Window attention (ws=14) for most layers; global attention at layers `[7, 15, 23, 31]`.  
`image_size=1024`, `patch_size=16` → **64×64 = 4 096 tokens**.  
Neck: two Conv2d layers (no FPN).

In [11]:
@torch.no_grad()
def profile_sam1(enc, img_size, repeats, device, dtype):
    cfg         = enc.config
    n_layers    = cfg.num_hidden_layers
    global_idxs = set(cfg.global_attn_indexes)
    ws          = cfg.window_size
    H_tok = W_tok = img_size // cfg.patch_size

    embed_stats = Stats()
    neck_stats  = Stats()
    enc_stats   = Stats()
    layer_stats = [{c: Stats() for c in ("layer_norm1", "attn", "layer_norm2", "mlp")}
                   for _ in range(n_layers)]

    timers = []
    timers.append(HookTimer(enc.patch_embed, embed_stats, device))
    timers.append(HookTimer(enc.neck,        neck_stats,  device))
    timers.append(HookTimer(enc,             enc_stats,   device))
    for i, layer in enumerate(enc.layers):
        s = layer_stats[i]
        for comp in ("layer_norm1", "attn", "layer_norm2", "mlp"):
            timers.append(HookTimer(getattr(layer, comp), s[comp], device))

    dummy = torch.randn(1, 3, img_size, img_size, device=device, dtype=dtype)
    for _ in range(2):
        enc(pixel_values=dummy)
    torch.cuda.synchronize(device)

    for _ in range(repeats):
        enc(pixel_values=torch.randn_like(dummy))

    for t in timers:
        t.remove()

    total_ms = enc_stats.mean
    win_idx  = [i for i in range(n_layers) if i not in global_idxs]
    glob_idx = [i for i in range(n_layers) if i in global_idxs]
    n_windows = (H_tok // ws) * (W_tok // ws)
    glob_seq  = H_tok * W_tok

    W = 84
    print(f"\n{'='*W}")
    print(f"SAM1  ({SAM1_MODEL})  |  Plain ViT  |  repeats={repeats}")
    print(f"  hidden={cfg.hidden_size}  heads={cfg.num_attention_heads}  layers={n_layers}  "
          f"patch={cfg.patch_size}  window_ws={ws}  global={sorted(global_idxs)}")
    print(f"  img={img_size}px  tokens={H_tok}\u00d7{W_tok}={H_tok*W_tok}")
    print(f"{'='*W}")
    print(f"  {'Component':<32} {'Mean ms':>9} {'Min ms':>9} {'Max ms':>9} {'Share':>7}")
    sep = f"  {'-'*(W-4)}"
    print(sep)

    def row(name, s, indent=0):
        pad = "  " * indent
        if isinstance(s, Stats):
            mean, mn, mx = s.mean, s.mn, s.mx
        else:
            mean = mn = mx = s
        pct = 100 * mean / total_ms if total_ms else 0
        print(f"  {pad}{name:<{32-2*indent}} {mean:>9.2f} {mn:>9.2f} {mx:>9.2f} {pct:>6.1f}%")

    row("PatchEmbed", embed_stats)
    print()

    for label, idxs, seq, n_wins in [
        (f"Window-attn  [{len(win_idx)} layers, ws={ws}]", win_idx,  ws*ws,    n_windows),
        (f"Global-attn  [{len(glob_idx)} layers]",          glob_idx, glob_seq, 1        ),
    ]:
        group_mean = sum(
            sum(layer_stats[i][c].mean for c in ("layer_norm1","attn","layer_norm2","mlp"))
            for i in idxs
        )
        pct = 100 * group_mean / total_ms
        avg = group_mean / max(len(idxs), 1)
        print(f"  {label:<38} {group_mean:>9.2f}  avg/layer={avg:.2f}ms  {pct:.1f}%")
        print(f"    attn_map: ({n_wins}, {cfg.num_attention_heads}, {seq}, {seq})")
        for comp in ("layer_norm1", "attn", "layer_norm2", "mlp"):
            cs = sum(layer_stats[i][comp].mean for i in idxs)
            cm = sum(layer_stats[i][comp].mn   for i in idxs)
            cx = sum(layer_stats[i][comp].mx   for i in idxs)
            ca = cs / max(len(idxs), 1)
            pp = 100 * cs / total_ms
            print(f"    {'└─ '+comp:<28} {cs:>9.2f} {cm:>9.2f} {cx:>9.2f} {pp:>6.1f}%  "
                  f"(avg/layer={ca:.3f}ms)")
        print()

    row("Neck  (Conv2d ×2)", neck_stats)
    print(sep)
    print(f"  {'TOTAL':<32} {total_ms:>9.2f}")
    print(f"  Throughput: {1000/total_ms:.2f} fps")

    print(f"\n  Window vs Global attention  (avg per layer):")
    print(f"  {'Component':<16} {'Window ms':>12} {'Global ms':>12}  Ratio")
    for comp in ("layer_norm1", "attn", "layer_norm2", "mlp"):
        w_avg = sum(layer_stats[i][comp].mean for i in win_idx)  / max(len(win_idx), 1)
        g_avg = sum(layer_stats[i][comp].mean for i in glob_idx) / max(len(glob_idx), 1)
        ratio = g_avg / w_avg if w_avg > 0 else float("nan")
        print(f"  {comp:<16} {w_avg:>12.3f} {g_avg:>12.3f}  {ratio:>6.1f}×")

    return dict(
        model="SAM1", total_ms=total_ms,
        embed_ms=embed_stats.mean, neck_ms=neck_stats.mean,
        n_tokens=H_tok*W_tok, n_layers=n_layers,
        win_attn_ms=sum(layer_stats[i]["attn"].mean for i in win_idx),
        glob_attn_ms=sum(layer_stats[i]["attn"].mean for i in glob_idx),
    )

sam1_results = profile_sam1(sam1_enc, SAM1_IMSIZE, REPEATS, DEVICE, DTYPE)


NameError: name 'sam1_enc' is not defined

## 3. SAM2 — Hiera Image Encoder

Architecture: **hierarchical ViT (Hiera)** — four stages with progressive downsampling.  
Patch embed: Conv2d 7×7, stride 4 → `(256×256)` at `embed_dim=144`.  
Stages (large): `[2, 6, 36, 4]` blocks with window sizes `[8, 4, 14, 7]`.  
Last `global_att_blocks` in stage 3 + all of stage 4 use **global** attention.  
Neck: 4-scale FPN → four `(256, H, W)` feature maps.

> **Mock mode**: no SAM2 install needed — mock weights from `profile_sam2_image_encoder.py`.


In [5]:
import sys
sys.path.insert(0, '/workspace/Block-Sparse-Attention')
from profile_sam2_image_encoder import (
    HieraImageEncoder, MODEL_CONFIGS,
    profile_image_encoder as _sam2_profiler,
)

_size_key = SAM2_MODEL.rsplit("-", 1)[-1].replace("-plus", "_plus")
_size_map  = {"tiny": "tiny", "small": "small", "large": "large",
              "base_plus": "base_plus", "base-plus": "base_plus"}
sam2_size  = _size_map.get(_size_key, "large")
sam2_cfg   = MODEL_CONFIGS[sam2_size]

print(f"Building mock SAM2-Hiera-{sam2_size}  (dtype={DTYPE}) ...")
sam2_enc = HieraImageEncoder(sam2_cfg, dtype=DTYPE).to(DEVICE).eval()
print(f"  stages={sam2_cfg['stages']}  embed_dim={sam2_cfg['embed_dim']}")
print(f"  window_spec={sam2_cfg['window_spec']}  global_att_blocks={sam2_cfg['global_att_blocks']}")
print(f"  neck_out_dim={sam2_cfg['neck_out_dim']}")


Building mock SAM2-Hiera-large  (dtype=torch.bfloat16) ...
  stages=(2, 6, 36, 4)  embed_dim=144
  window_spec=(8, 4, 14, 7)  global_att_blocks=(23, 33, 43)
  neck_out_dim=256


In [6]:
with torch.no_grad():
    _sam2_profiler(sam2_enc, torch.device(DEVICE), DTYPE, SAM2_IMSIZE, REPEATS)

# End-to-end timing for the comparison table
_s2 = Stats()
_t2 = HookTimer(sam2_enc, _s2, DEVICE)
with torch.no_grad():
    _d2 = torch.randn(1, 3, SAM2_IMSIZE, SAM2_IMSIZE, device=DEVICE, dtype=DTYPE)
    for _ in range(REPEATS):
        sam2_enc(_d2)
_t2.remove()

_stages = sam2_cfg["stages"]
_n_tok  = (SAM2_IMSIZE // 4) // (2 ** (len(_stages) - 1))   # rough final res
sam2_results = dict(
    model="SAM2", total_ms=_s2.mean,
    embed_ms=0.0, neck_ms=0.0,          # detailed breakdown lives in _sam2_profiler output
    n_tokens=f"multi-scale ({SAM2_IMSIZE//4}→{_n_tok})",
    n_layers=sum(_stages),
)
print(f"\nSAM2 end-to-end: {_s2.mean:.2f} ms  ({1000/_s2.mean:.2f} fps)")



SAM2 Hiera Image Encoder Profile  |  img=1024×1024  |  repeats=10
Component                      Mean ms    Min ms    Max ms    Mem MB   Share%
────────────────────────────────────────────────────────────────────────────
PatchEmbed                        1.22      1.14      1.47      54.5     1.0%

Stage 1  [2 blocks]               7.08      6.96      7.84     180.0     6.0%
    attn_map shape           (1024, 2, 64, 64)  [window(ws=8)]
  └─ norm1                       0.69      0.66      0.81               0.6%  (avg/blk=0.345ms)
  └─ attn                        3.39      3.37      3.42               2.9%  (avg/blk=1.693ms)
  └─ norm2                       0.68      0.66      0.74               0.6%  (avg/blk=0.339ms)
  └─ mlp                         2.44      2.43      2.47               2.1%  (avg/blk=1.221ms)
  └─ downsample (conv)           0.33      0.32      0.34               0.3%

Stage 2  [6 blocks]              10.44     10.10     10.86      99.0     8.9%
    attn_map shape

## 4. SAM3 — ViT + RoPE Image Encoder

Architecture: **32-layer ViT** with **Rotary Position Embeddings (RoPE)**.  
`image_size=1008`, `patch_size=14` → **72×72 = 5 184 tokens**.  
Window attention (ws=24) for most layers; global attention at `[7, 15, 23, 31]`.  
Neck: 4-scale FPN (same as SAM2).

> **Mock mode** (`SAM3_MOCK=True`): random weights via `build_mock_vision_encoder`.  
> Set `HF_TOKEN` and `SAM3_MOCK=False` to use real weights from `facebook/sam3`.


In [7]:
from profile_sam3_hf_image_encoder import (
    build_mock_vision_encoder  as _build_sam3_mock,
    profile_vision_encoder     as _sam3_profiler,
)

if not SAM3_MOCK:
    try:
        from profile_sam3_hf_image_encoder import load_vision_encoder_pretrained as _load_sam3
        sam3_enc = _load_sam3(SAM3_MODEL, HF_TOKEN, torch.device(DEVICE), DTYPE)
    except Exception as _e:
        print(f"[WARN] Real SAM3 load failed ({_e}). Falling back to mock.")
        sam3_enc = _build_sam3_mock(torch.device(DEVICE), DTYPE)
else:
    sam3_enc = _build_sam3_mock(torch.device(DEVICE), DTYPE)

sam3_cfg = sam3_enc.backbone.config
print(f"  hidden={sam3_cfg.hidden_size}  heads={sam3_cfg.num_attention_heads}  "
      f"layers={sam3_cfg.num_hidden_layers}  patch={sam3_cfg.patch_size}  "
      f"window={sam3_cfg.window_size}  global={sam3_cfg.global_attn_indexes}")


Building mock SAM3 vision encoder from default config (random weights) ...
  hidden=1024  heads=16  layers=32  patch=14  window=24  global=[7, 15, 23, 31]


In [8]:
with torch.no_grad():
    _sam3_profiler(sam3_enc, SAM3_IMSIZE, REPEATS, torch.device(DEVICE), DTYPE)

# End-to-end timing for the comparison table
_s3 = Stats()
_t3 = HookTimer(sam3_enc, _s3, DEVICE)
with torch.no_grad():
    _d3 = torch.randn(1, 3, SAM3_IMSIZE, SAM3_IMSIZE, device=DEVICE, dtype=DTYPE)
    for _ in range(REPEATS):
        sam3_enc(pixel_values=_d3)
_t3.remove()

_c3   = sam3_enc.backbone.config
_H3   = SAM3_IMSIZE // _c3.patch_size
sam3_results = dict(
    model="SAM3", total_ms=_s3.mean,
    embed_ms=0.0, neck_ms=0.0,
    n_tokens=_H3 * _H3,
    n_layers=_c3.num_hidden_layers,
)
print(f"\nSAM3 end-to-end: {_s3.mean:.2f} ms  ({1000/_s3.mean:.2f} fps)")


Sam3ViTModel(
  (embeddings): Sam3ViTEmbeddings(
    (patch_embeddings): Sam3ViTPatchEmbeddings(
      (projection): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (layer_norm): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
  (layers): ModuleList(
    (0-31): 32 x Sam3ViTLayer(
      (layer_norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (rotary_emb): Sam3ViTRotaryEmbedding()
      (attention): Sam3ViTRoPEAttention(
        (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (o_proj): Linear(in_features=1024, out_features=1024, bias=True)
      )
      (layer_norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Sam3MLP(
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features

## 5. Side-by-Side Comparison

Aggregate summary across all three encoders.  
Attention cost ratios show how much **more expensive** global attention is vs window attention per layer.


In [14]:
results = [ sam2_results, sam3_results]

W = 80
print(f"\n{'='*W}")
print("SAM Encoder Comparison  (batch=1, bfloat16)")
print(f"{'='*W}")
print(f"  {'Model':<10} {'Total ms':>10} {'FPS':>8} {'Layers':>8} {'Tokens':>18}")
print(f"  {'-'*(W-4)}")
for r in results:
    fps     = 1000 / r['total_ms']
    layers  = str(r.get('n_layers', '—'))
    tokens  = str(r.get('n_tokens', '—'))
    print(f"  {r['model']:<10} {r['total_ms']:>10.2f} {fps:>8.2f} {layers:>8} {tokens:>18}")


print(f"\n  Key insight for block-sparse attention:")
# H1 = SAM1_IMSIZE // sam1_enc.config.patch_size
# ws = sam1_enc.config.window_size
# n_windows = (H1 // ws) ** 2
# print(f"    SAM1 global tokens  : {H1}×{H1} = {H1*H1}")
# print(f"    SAM1 window tokens  : {ws}×{ws} = {ws*ws}  (n_windows={n_windows})")
H3 = SAM3_IMSIZE // sam3_enc.backbone.config.patch_size
ws3 = sam3_enc.backbone.config.window_size
n_windows3 = (H3 // ws3) ** 2
print(f"    SAM3 global tokens  : {H3}×{H3} = {H3*H3}")
print(f"    SAM3 window tokens  : {ws3}×{ws3} = {ws3*ws3}  (n_windows={n_windows3})")
print(f"    Block-sparse target : replace {H3*H3}-token global attn with ~{ws3*ws3}-token windows")



SAM Encoder Comparison  (batch=1, bfloat16)
  Model        Total ms      FPS   Layers             Tokens
  ----------------------------------------------------------------------------
  SAM2           127.26     7.86       48 multi-scale (256→32)
  SAM3           198.02     5.05       32               5184

  Key insight for block-sparse attention:
    SAM3 global tokens  : 72×72 = 5184
    SAM3 window tokens  : 24×24 = 576  (n_windows=9)
    Block-sparse target : replace 5184-token global attn with ~576-token windows
